# 04 · LIME — Local Interpretable Model-agnostic Explanations

In [ ]:
!pip install shap lime matplotlib seaborn pandas numpy scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.datasets import load_breast_cancer
import shap
import lime
import lime.lime_tabular

np.random.seed(42)


In [ ]:
# Load the Breast Cancer Wisconsin dataset (used throughout this course)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Train a Random Forest — a strong but harder-to-interpret "black box" model
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")


## LIME Explanation for a Single Prediction

In [ ]:
# Initialize LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=data.feature_names,
    class_names=['Malignant', 'Benign'],
    mode='classification'
)

# Explain single prediction
patient_idx = 5
exp = lime_explainer.explain_instance(
    X_test.iloc[patient_idx].values,
    rf.predict_proba,
    num_features=10
)

# Show explanation
exp.show_in_notebook(show_table=True)

# Save as matplotlib figure
fig = exp.as_pyplot_figure()
plt.title(f"LIME Explanation - Patient {patient_idx}")
plt.tight_layout()
plt.show()

# Get explanation as list
lime_list = exp.as_list()
print("\nLIME top features (positive=supports Benign, negative=supports Malignant):")
for feature, weight in lime_list[:5]:
    print(f"  {feature}: {weight:.3f}")
